In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

E_beam = 80e-3  # 100 mJ in a pulse
dx = 4*8.73e-6  # pixel size of the spiricon camera (9 microns) - and 4 pixel binning!
A_pix_cm = dx**2 * 1e4  # pixel area in cm^2

def gaussian(x, a, x0, sigma):
    return a * np.exp(-(x - x0)**2 / (2 * sigma**2))

def fit_gaussian(data):
    x = np.arange(len(data))
    initial_guess = [np.max(data), len(data) // 2, len(data) / 10]
    try:
        popt, _ = curve_fit(gaussian, x, data, p0=initial_guess)
        return popt
    except RuntimeError:
        return None

def plot_ascii_csv_files_with_peak(folder_path,ascii_csv_files):

    max_fluences = []
    h_diams = []
    v_diams = []

    # Plot each ASCII CSV file
    for csv_file in ascii_csv_files:
        file_path = os.path.join(folder_path, csv_file)
        try:
            # Load the CSV file into a numpy array
            data = np.genfromtxt(file_path, delimiter=',')

            # Replace NaN or inf values with zeros
            data = np.nan_to_num(data, nan=0.0, posinf=0.0, neginf=0.0)

            # Find the peak value in the data
            peak_value = np.max(data)

            # Replace all pixel values less than 1% of the peak value with zeros
            threshold = 0.01 * peak_value
            data[data < threshold] = 0

            sum_vals = np.sum(data)  # sum of all the pixel counts
            E_per_pixcount = E_beam / sum_vals  # energy per one pixel count

            # Find the peak in the data
            max_loc = np.unravel_index(np.argmax(data), data.shape)

            # Create a mask for a circle with radius 5 pixels around the peak location
            y, x = np.ogrid[:data.shape[0], :data.shape[1]]
            center_y, center_x = max_loc
            mask = (x - center_x)**2 + (y - center_y)**2 <= 5**2

            # Apply the mask to the data to get the values within the circle
            values_in_circle = data[mask]

            # Calculate the average value within the peak circle
            avg_peak = np.mean(values_in_circle)

            #save the peak fluence
            max_fluences.append(avg_peak * E_per_pixcount / A_pix_cm)

            # Crop the data to a 600x600 pixel square centered on the peak
            center_y, center_x = max_loc
            start_x = max(center_x - 300, 0)
            end_x = min(center_x + 300, data.shape[1])
            start_y = max(center_y - 300, 0)
            end_y = min(center_y + 300, data.shape[0])
            cropped_data = data[start_y:end_y, start_x:end_x]

            # Calculate the fluence
            fluence_map = cropped_data * E_per_pixcount / A_pix_cm

            # Extract the lineouts for gaussian fitting (sum along axis)
            horizontal_lineout = np.mean(fluence_map,axis=0)
            vertical_lineout = np.mean(fluence_map,axis=1)

            # Fit Gaussian to the lineouts
            h_fit_params = fit_gaussian(horizontal_lineout)
            v_fit_params = fit_gaussian(vertical_lineout)

            # Calculate the 1/e^2 diameters
            if h_fit_params is not None:
                h_fwhm = 2 * np.sqrt(2*np.log(2)) * h_fit_params[2] #calculate FWHM from sigma
                h_esqr_mm = 1.699 * h_fwhm * dx * 1e3  # convert FWHM to 1/e^2 diameter, and into mm
                h_diams.append(h_esqr_mm)
            else:
                h_esqr_mm = None

            if v_fit_params is not None:
                v_fwhm = 2 * np.sqrt(2) * np.sqrt(np.log(2)) * v_fit_params[2] #calculate FWHM from sigma
                v_esqr_mm = 1.699 * v_fwhm * dx * 1e3 # convert FWHM to 1/e^2 diameter, and into mm
                v_diams.append(v_esqr_mm)
            else:
                v_esqr_mm = None

            # Create mm scales for the axes
            x_mm = np.arange(fluence_map.shape[1]) * dx * 1e3
            y_mm = np.arange(fluence_map.shape[0]) * dx * 1e3

            # Plot the data using matplotlib
            fig, ax = plt.subplots(figsize=(6, 6))

            # Plot the fluence map
            im = ax.imshow(fluence_map, extent=[x_mm[0], x_mm[-1], y_mm[-1], y_mm[0]], cmap='RdPu', vmin=0, vmax=1)
            ax.scatter((center_x-start_x) * dx * 1e3, (center_y-start_y) * dx * 1e3, color='red', s=100, marker='x')
            ax.set_xlabel('X axis (mm)')
            ax.set_ylabel('Y axis (mm)')
            plt.colorbar(im, ax=ax, label='J/cm^2')

            lfactor = 50
            # Overlay the lineouts
            ax.plot(x_mm, horizontal_lineout * lfactor, color='blue', label='Horizontal Lineout')
            ax.plot(vertical_lineout * lfactor, y_mm, color='green', label='Vertical Lineout')

            # Overlay the Gaussian fits
            if h_fit_params is not None:
                ax.plot(x_mm, gaussian(np.arange(len(x_mm)), *h_fit_params)*lfactor, color='cyan', linestyle='--', label='Horizontal Fit')
            if v_fit_params is not None:
                ax.plot(gaussian(np.arange(len(y_mm)), *v_fit_params) * lfactor, y_mm, color='yellow', linestyle='--', label='Vertical Fit')

            ax.legend()

            plt.tight_layout()
            plt.show()

            print(f"File: {csv_file}")
            print(f"Horizontal 1/e^2 Diameter: {h_esqr_mm:.4f} mm" if h_esqr_mm else "Horizontal fit failed")
            print(f"Vertical 1/e^2 Diameter: {v_esqr_mm:.4f} mm" if v_esqr_mm else "Vertical fit failed")

        except ValueError as ve:
            print(f"ValueError: Could not process {csv_file}: {ve}")
        except Exception as e:
            print(f"Could not process {csv_file}: {e}")

    return max_fluences,h_diams,v_diams

# Example usage:
folder_path_1 = '/Users/sebastiankalos/Documents/OXFORD/CALA/Expt_June2024/21062024/grid1'
ascii_csv_files_1 = ['6_0001.ascii.csv',
                        '7_0001.ascii.csv',
                        '8_0001.ascii.csv',
                        '9_0001.ascii.csv',
                        '10_0001.ascii.csv',
                        '11_0001.ascii.csv',
                        '12_0001.ascii.csv',
                        '13_0001.ascii.csv',
                        '14_0001.ascii.csv',
                        '15_0001.ascii.csv',
                        '16_0001.ascii.csv',
                        '17_0001.ascii.csv']
max_fluences_1,h_diams_1,v_diams_1 = plot_ascii_csv_files_with_peak(folder_path_1, ascii_csv_files_1)
